# Brazilian E-Commerce Public Dataset by Olist
## SQL Exploratory Data Analysis (EDA)

## 1. Project Overview
This project explores Olist’s Brazilian e-commerce data using SQL. The analysis examines marketing lead acquisition and conversion, seller acquisition and activation, seller revenue performance, and marketplace sales and delivery operations. The goal is to summarize patterns in the available data and highlight findings supported by the SQL analysis.

## 2. Dataset Overview
The SQL analysis references the following tables: `marketing_qualified_leads`, `closed_deals`, `orders`, `order_items`, and `customers`. The queries combine the marketing and marketplace tables to examine the lead-to-seller process and marketplace performance.


## 3. DataBase Connection


In [6]:
import psycopg2
import pandas as pd
def connect_db():
    conn = psycopg2.connect(
        host="localhost",
        database="Marketing Funnel by Olist",
        user="postgres",
        password="2003",
        port="5432"
    )
    cursor = conn.cursor()
    return conn, cursor

def sql_to_df(sql):
    conn, cursor = connect_db()
    cursor.execute(sql)
    result = cursor.fetchall()
    df = pd.DataFrame(result, columns=[desc[0] for desc in cursor.description])
    cursor.close()
    conn.close()
    return df

## 4. SQL Analysis

### Phase 1 — Marketing Lead Acquisition


#### 1. Initial Data Inspection — Marketing Qualified Leads


In [ ]:
SELECT *
FROM marketing_qualified_leads;

#### 2. Which Marketing Sources Generate the Highest Number of Leads?


In [7]:
sql_1 = ''' WITH lead_by_origin AS (
    SELECT
        COALESCE(origin, 'unknown') AS origin,
        COUNT(DISTINCT mql_id) AS total_leads
    FROM marketing_qualified_leads
    GROUP BY 1
)
SELECT
    origin,
    total_leads,
    ROUND(
        100.0 * total_leads / SUM(total_leads) OVER(),
        2
    ) AS lead_percentage
FROM lead_by_origin
ORDER BY lead_percentage DESC;'''

result = sql_to_df(sql_1)
result

,origin,total_leads,lead_percentage
0,organic_search,2296,28.70
1,paid_search,1586,19.83
2,social,1350,16.88
3,unknown,1159,14.49
4,direct_traffic,499,6.24
5,email,493,6.16
6,referral,284,3.55
7,other,150,1.88
8,display,118,1.48
9,other_publicities,65,0.81


**Key Insights**

- Organic Search is the largest source of leads, contributing 28.70%, followed by Paid Search (19.83%) and Social (16.88%). Together, these three channels account for 65.41% of total leads.
- Unknown sources represent 14.49% of leads, indicating a gap in marketing source tracking.


#### 3. Which Marketing Sources Generate the Highest Lead Conversion Rate?


In [8]:
sql_2 = '''WITH leads AS (
    SELECT DISTINCT
        mql_id,
        COALESCE(origin, 'unknown') AS origin
    FROM marketing_qualified_leads
    WHERE mql_id IS NOT NULL
),
converted_leads AS (
    SELECT DISTINCT
        mql_id
    FROM closed_deals
    WHERE mql_id IS NOT NULL
)
SELECT
    l.origin,
    COUNT(DISTINCT l.mql_id) AS total_leads,
    COUNT(DISTINCT c.mql_id) AS converted_leads,
    ROUND(
        100.0 * COUNT(DISTINCT c.mql_id) /
        COUNT(DISTINCT l.mql_id),
        2
    ) AS conversion_rate
FROM leads l
LEFT JOIN converted_leads c
    ON l.mql_id = c.mql_id
GROUP BY l.origin
ORDER BY conversion_rate DESC;'''
result = sql_to_df(sql_2)
result

,origin,total_leads,converted_leads,conversion_rate
0,unknown,1159,193,16.65
1,paid_search,1586,195,12.30
2,organic_search,2296,271,11.80
3,direct_traffic,499,56,11.22
4,referral,284,24,8.45
5,social,1350,75,5.56
6,display,118,6,5.08
7,other_publicities,65,3,4.62
8,email,493,15,3.04
9,other,150,4,2.67


**Key Insights**

- Organic Search generates the highest number of converted leads (271), making it the leading source in terms of successful deals.
- Paid Search achieves the highest conversion rate among identifiable major marketing channels at 12.30%, slightly above Organic Search (11.80%).
- Social Media generates a significant share of leads but has a relatively low conversion rate of 5.56%, suggesting an opportunity to improve lead quality or targeting.
- Unknown sources have the highest conversion rate (16.65%), but missing source attribution makes it difficult to identify which channels drive this performance.


#### 4. How Does Lead Conversion Rate Vary Across Months?


In [9]:
sql_3 = '''SELECT
    DATE_TRUNC('month', l.first_contact_date)::date AS month,
    COUNT(DISTINCT l.mql_id) AS total_leads,
    COUNT(DISTINCT c.mql_id) AS converted_leads,
    ROUND(
        100.0 * COUNT(DISTINCT c.mql_id) /
        COUNT(DISTINCT l.mql_id),
        2
    ) AS monthly_conversion_rate
FROM marketing_qualified_leads l
LEFT JOIN closed_deals c
    ON l.mql_id = c.mql_id
WHERE l.first_contact_date IS NOT NULL
GROUP BY 1
ORDER BY month;'''
result = sql_to_df(sql_3)
result

,month,total_leads,converted_leads,monthly_conversion_rate
0,2017-06-01,4,0,0.00
1,2017-07-01,239,2,0.84
2,2017-08-01,386,9,2.33
3,2017-09-01,312,7,2.24
4,2017-10-01,416,14,3.37
5,2017-11-01,445,18,4.04
6,2017-12-01,200,11,5.50
7,2018-01-01,1141,152,13.32
8,2018-02-01,1028,149,14.49
9,2018-03-01,1174,167,14.22


**Key Insights**

- Lead conversion rate improved significantly from 0.84% in July 2017 to over 14% in early 2018, indicating a substantial improvement in lead conversion performance.
- Conversion rate peaked at 14.49% in February 2018, then declined to 9.98% in May 2018, despite maintaining a relatively high lead volume. This may indicate a recent decline in conversion effectiveness.


#### 5. How Many Days Does It Take to Convert a Qualified Lead into a Closed Deal?


In [10]:
sql_4 = '''WITH converted_leads AS (
    SELECT
        m.mql_id,
        m.origin,
        m.first_contact_date,
        c.won_date
    FROM marketing_qualified_leads m
    JOIN closed_deals c
        ON m.mql_id = c.mql_id
),
sales_cycle AS (
    SELECT
        origin,
        won_date - first_contact_date AS days_to_close
    FROM converted_leads
)
SELECT
    origin,
    COUNT(*) AS converted_leads,
    AVG(days_to_close) AS avg_days_to_close,
    PERCENTILE_CONT(0.5) WITHIN GROUP(
        ORDER BY days_to_close
    ) AS median_days_to_close,
    MIN(days_to_close) AS min_days_to_close,
    MAX(days_to_close) AS max_days_to_close
FROM sales_cycle
WHERE days_to_close >= INTERVAL '0 days'
GROUP BY origin
ORDER BY avg_days_to_close;'''

result = sql_to_df(sql_4)
result

,origin,converted_leads,avg_days_to_close,median_days_to_close,min_days_to_close,max_days_to_close
0,display,6,10 days 22:07:33.333333,9 days 04:03:22.500000,0 days 16:07:32,25 days 13:30:05
1,other,4,15 days 18:24:04,9 days 13:22:35.500000,1 days 03:00:00,42 days 19:51:05
2,direct_traffic,56,31 days 18:25:30.696429,10 days 13:49:40.500000,0 days 17:13:16,306 days 16:35:47
3,referral,24,33 days 05:49:54.125000,19 days 08:53:54.500000,0 days 16:43:53,132 days 14:58:50
4,other_publicities,3,40 days 01:48:40.333333,35 days 19:00:03,35 days 18:14:08,48 days 16:11:50
5,unknown,179,41 days 22:37:50.251397,11 days 19:51:21,0 days 03:00:00,369 days 20:24:28
6,None,14,49 days 21:50:02.857143,8 days 02:00:53.500000,0 days 18:57:37,427 days 13:14:37
7,organic_search,270,50 days 20:28:24.944444,15 days 05:55:43.500000,0 days 02:00:00,353 days 14:41:13
8,email,15,52 days 18:33:24.800000,21 days 03:00:00,1 days 21:38:40,226 days 17:17:42
9,paid_search,195,57 days 06:35:03.615385,15 days 16:11:25,0 days 03:00:00,373 days 18:32:49


**Key Insights**

- Direct Traffic has the shortest sales cycle among the major channels, with an average of approximately 32 days and a median of 11 days.
- Social Media has the longest sales cycle, averaging approximately 61 days, with a median of 31 days.
- The large gap between average and median conversion times across several channels suggests that a small number of leads take significantly longer to convert.
- Display has the shortest average conversion time (around 11 days), but the sample includes only 6 converted leads, so this result should be interpreted cautiously.
- =====================================================


### Phase 2 — Seller Acquisition & Activation


#### 6. Which Business Segments Have the Highest Number of Acquired Sellers?


In [11]:
sql_5 = '''SELECT
    COALESCE(business_segment, 'unknown') AS business_segment,
    COUNT(DISTINCT seller_id) AS total_sellers,
    ROUND(
        100.0 * COUNT(DISTINCT seller_id) /
        SUM(COUNT(DISTINCT seller_id)) OVER(),
        2
    ) AS seller_percentage
FROM closed_deals
GROUP BY 1
ORDER BY total_sellers DESC;'''
result = sql_to_df(sql_5)
result

,business_segment,total_sellers,seller_percentage
0,home_decor,105,12.47
1,health_beauty,93,11.05
2,car_accessories,77,9.14
3,household_utilities,71,8.43
4,construction_tools_house_garden,69,8.19
5,audio_video_electronics,64,7.60
6,computers,34,4.04
7,pet,30,3.56
8,food_supplement,28,3.33
9,food_drink,26,3.09


**Key Insights**

- Home Decor is the largest business segment, accounting for 12.47% of acquired sellers, followed by Health & Beauty (11.05%) and Car Accessories (9.14%).
- The top five business segments account for approximately 49.28% of acquired sellers, indicating that seller acquisition is distributed across several business categories rather than dominated by a single segment.


#### 7. Which Business Segments Have the Shortest Average Time to Close a Deal?


In [12]:
sql_6 = '''SELECT
    COALESCE(c.business_segment, 'unknown') AS business_segment,
    COUNT(DISTINCT c.mql_id) AS converted_leads,
    AVG(c.won_date - l.first_contact_date) AS avg_days_to_close
FROM marketing_qualified_leads l
JOIN closed_deals c
    ON l.mql_id = c.mql_id
WHERE l.first_contact_date IS NOT NULL
    AND c.won_date IS NOT NULL
    AND c.won_date >= l.first_contact_date
GROUP BY 1
ORDER BY avg_days_to_close ASC;'''
result = sql_to_df(sql_6)
result

,business_segment,converted_leads,avg_days_to_close
0,religious,1,4 days 16:27:35
1,bags_backpacks,22,20 days 04:54:55
2,home_office_furniture,14,22 days 09:45:08.142857
3,air_conditioning,3,22 days 16:13:27.666667
4,games_consoles,2,26 days 04:53:57.500000
5,sports_leisure,25,27 days 08:29:29.080000
6,pet,30,29 days 20:31:59.633333
7,phone_mobile,13,30 days 03:56:51.923077
8,food_drink,26,32 days 05:40:31.346154
9,health_beauty,92,36 days 02:32:11.836957


**Key Insights**

- Bags & Backpacks has the shortest average sales cycle among segments with a meaningful number of converted leads, at approximately 20 days (22 converted leads).
- Health & Beauty combines a relatively large number of converted leads (92) with an average conversion time of approximately 36 days.
- Home Decor and Car Accessories take approximately 51 days to close a deal, despite being among the largest segments by seller acquisition.
- Some segments show significantly longer conversion times, such as Audio Video Electronics (approximately 64 days) and Watches (approximately 92 days).


#### 8. How Many Acquired Sellers Became Active on the Marketplace?


In [14]:
sql_7 = '''SELECT
    COUNT(DISTINCT c.seller_id) AS total_sellers,
    COUNT(DISTINCT oi.seller_id) AS active_sellers,
    ROUND(
        100.0 * COUNT(DISTINCT oi.seller_id) /
        COUNT(DISTINCT c.seller_id),
        2
    ) AS percentage
FROM closed_deals c
LEFT JOIN order_items oi
    ON c.seller_id = oi.seller_id;'''
result = sql_to_df(sql_7)
result

,total_sellers,active_sellers,percentage
0,842,380,45.13


**Key Insights**

- Only 45.13% of acquired sellers (380 out of 842) became active on the marketplace, indicating that more than half of acquired sellers did not generate any orders.
- This highlights a potential gap between seller acquisition and marketplace activation, suggesting a need to improve the onboarding and activation process.


#### 9. What Percentage of Acquired Sellers Successfully Fulfilled Customer Orders?


In [15]:
sql_8 = '''SELECT
    COUNT(DISTINCT c.seller_id) AS total_sellers,
    COUNT(DISTINCT CASE
        WHEN o.order_status = 'delivered'
        THEN oi.seller_id
    END) AS sellers_with_delivered,
    ROUND(
        100.0 * COUNT(DISTINCT CASE
            WHEN o.order_status = 'delivered'
            THEN oi.seller_id
        END) / COUNT(DISTINCT c.seller_id),
        2
    ) AS percentage
FROM closed_deals c
LEFT JOIN order_items oi
    ON c.seller_id = oi.seller_id
LEFT JOIN orders o
    ON oi.order_id = o.order_id;'''
result = sql_to_df(sql_8)
result

,total_sellers,sellers_with_delivered,percentage
0,842,376,44.66


**Key Insights**

- Only 44.66% of acquired sellers (376 out of 842) successfully fulfilled customer orders, meaning more than half of acquired sellers did not generate delivered orders.
- The percentage is slightly lower than the seller activation rate (45.13%), indicating that a small number of active sellers had no successfully delivered orders.
- =====================================================


### Phase 3 — Seller Performance


#### 10. How Much Revenue Does Each Seller Generate?


In [16]:
sql_9 = '''SELECT
    c.seller_id,
    COUNT(DISTINCT o.order_id) AS count_orders,
    SUM(oi.price) AS revenue
FROM closed_deals c
JOIN order_items oi
    ON c.seller_id = oi.seller_id
JOIN orders o
    ON oi.order_id = o.order_id
WHERE o.order_status = 'delivered'
GROUP BY 1
ORDER BY revenue DESC;'''
result = sql_to_df(sql_9)
result

,seller_id,count_orders,revenue
0,7d13fca15225358621be4086e1eb0964,558,112436.18
1,ba90964cff9b9e0e6f32b23b82465f7b,38,44212.50
2,c70c1b0d8ca86052f45a432a38b73958,260,36289.15
3,612170e34b97004b3ba37eae81836b4c,107,23065.02
4,6061155addc1e54b4cfb51c1c2a32ad8,28,15633.40
...,...,...,...
371,5e063e85d44b0f5c3e6ec3131103a57e,1,22.90
372,bc8c8d665ec4664d286be0d521722b19,1,22.50
373,455befea1bb853c75ccbca94cff26e5c,1,19.99
374,bb7ad8a45c027be8ab075b8e465f8ca0,2,19.98


#### 11. Which Business Segments Generate the Highest Marketplace Revenue?


In [17]:
sql_10 = '''WITH acquired_sellers AS (
    SELECT DISTINCT
        seller_id,
        COALESCE(business_segment, 'unknown') AS business_segment
    FROM closed_deals
    WHERE seller_id IS NOT NULL
),
seller_performance AS (
    SELECT
        oi.seller_id,
        COUNT(DISTINCT o.order_id) AS delivered_orders,
        SUM(oi.price) AS product_revenue
    FROM order_items oi
    JOIN orders o
        ON oi.order_id = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY oi.seller_id
)
SELECT
    a.business_segment,
    COUNT(DISTINCT a.seller_id) AS acquired_sellers,
    COUNT(DISTINCT CASE
        WHEN sp.seller_id IS NOT NULL
        THEN a.seller_id
    END) AS sellers_with_delivered_orders,
    SUM(COALESCE(sp.delivered_orders, 0)) AS delivered_orders,
    SUM(COALESCE(sp.product_revenue, 0)) AS product_revenue,
    ROUND(
        AVG(COALESCE(sp.product_revenue, 0)),
        2
    ) AS avg_revenue_per_acquired_seller
FROM acquired_sellers a
LEFT JOIN seller_performance sp
    ON a.seller_id = sp.seller_id
GROUP BY a.business_segment
ORDER BY product_revenue DESC;'''
result = sql_to_df(sql_10)
result

,business_segment,acquired_sellers,sellers_with_delivered_orders,delivered_orders,product_revenue,avg_revenue_per_acquired_seller
0,watches,8,2,572,115901.08,14487.64
1,health_beauty,93,44,693,88924.34,956.18
2,household_utilities,71,44,498,51280.87,722.27
3,audio_video_electronics,64,31,244,49423.44,772.24
4,small_appliances,12,7,68,47674.87,3972.91
5,home_decor,105,43,356,43809.33,417.23
6,pet,30,17,247,40498.83,1349.96
7,construction_tools_house_garden,69,32,267,32007.07,463.87
8,car_accessories,77,30,147,30005.98,389.69
9,home_appliances,7,5,135,26241.17,3748.74


**Key Insights**

- Watches generates the highest product revenue at 115,901, despite having only 2 active sellers out of 8 acquired sellers, indicating a strong concentration of revenue among a small number of sellers.
- Health & Beauty has the highest number of delivered orders (693) and 44 active sellers, making it one of the strongest segments in terms of both seller participation and sales volume.
- Small Appliances achieves a high average revenue per acquired seller (3,972.91), followed by Home Appliances (3,748.74), although both segments have relatively small seller bases.
- Home Decor has the largest acquired seller base (105), but generates only 43,809 in product revenue, highlighting that a larger seller base does not necessarily translate into higher revenue.


#### 12. How Concentrated Is Marketplace Revenue Among Top Sellers?


In [18]:
sql_11 = '''SELECT
    *,
    SUM(revenue) OVER(ORDER BY revenue DESC) AS cumulative_revenue,
    ROUND(
        100.0 * SUM(revenue) OVER(ORDER BY revenue DESC) /
        SUM(revenue) OVER(),
        2
    ) AS cumulative_revenue_percentage,
    ROUND(
        100.0 * ROW_NUMBER() OVER(ORDER BY revenue DESC) /
        COUNT(*) OVER(),
        2
    ) AS sellers_percentage
FROM (
    SELECT
        c.seller_id,
        COUNT(DISTINCT o.order_id) AS count_orders,
        SUM(oi.price) AS revenue
    FROM closed_deals c
    JOIN order_items oi
        ON c.seller_id = oi.seller_id
    JOIN orders o
        ON oi.order_id = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY 1
) t
ORDER BY revenue DESC;'''
result = sql_to_df(sql_11)
result

,seller_id,count_orders,revenue,cumulative_revenue,cumulative_revenue_percentage,sellers_percentage
0,7d13fca15225358621be4086e1eb0964,558,112436.18,112436.18,16.91,0.27
1,ba90964cff9b9e0e6f32b23b82465f7b,38,44212.50,156648.68,23.56,0.53
2,c70c1b0d8ca86052f45a432a38b73958,260,36289.15,192937.83,29.02,0.80
3,612170e34b97004b3ba37eae81836b4c,107,23065.02,216002.85,32.49,1.06
4,6061155addc1e54b4cfb51c1c2a32ad8,28,15633.40,231636.25,34.84,1.33
...,...,...,...,...,...,...
371,5e063e85d44b0f5c3e6ec3131103a57e,1,22.90,664776.63,99.99,98.94
372,bc8c8d665ec4664d286be0d521722b19,1,22.50,664799.13,99.99,99.20
373,455befea1bb853c75ccbca94cff26e5c,1,19.99,664819.12,99.99,99.47
374,bb7ad8a45c027be8ab075b8e465f8ca0,2,19.98,664839.10,100.00,99.73


**Key Insights**

- =====================================================


### Phase 4 — Marketplace Sales & Operations


#### 13. What Happens to the Orders Placed on Olist?


In [19]:
sql_12 = '''SELECT
    order_status,
    count_orders,
    SUM(count_orders) OVER() AS total_orders,
    100.0 * count_orders / SUM(count_orders) OVER() AS percentage
FROM (
    SELECT
        order_status,
        COUNT(DISTINCT order_id) AS count_orders
    FROM orders
    GROUP BY order_status
) t
ORDER BY count_orders DESC;'''
result = sql_to_df(sql_12)
result

,order_status,count_orders,total_orders,percentage
0,delivered,96478,99441,97.0203437214026408
1,shipped,1107,99441,1.1132229161010046
2,canceled,625,99441,0.62851338984925734858
3,unavailable,609,99441,0.61242344706911636045
4,invoiced,314,99441,0.31576512706026689193
5,processing,301,99441,0.30269204855140233908
6,created,5,99441,0.00502810711879405879
7,approved,2,99441,0.00201124284751762352


**Key Insights**

- 97.02% of orders (96,478 out of 99,441) were successfully delivered, indicating a high overall order fulfillment rate.
- Only 1.24% of orders were canceled or unavailable combined, suggesting that order failures represent a relatively small share of total orders.
- The remaining orders were distributed across intermediate statuses, such as shipped, invoiced, and processing.


#### 14. How Is Olist's Sales Performance Evolving Over Time?


In [20]:
sql_13 = '''WITH orders_revenue AS (
    SELECT
        order_id,
        SUM(price) AS revenue
    FROM order_items
    GROUP BY order_id
),
orders_date AS (
    SELECT
        order_id,
        DATE_TRUNC('month', order_purchase_timestamp) AS month
    FROM orders
    WHERE order_status = 'delivered'
)
SELECT
    t2.month,
    COUNT(*) AS total_orders,
    SUM(t1.revenue) AS monthly_revenue,
    AVG(t1.revenue) AS AOV
FROM orders_revenue t1
JOIN orders_date t2
    ON t1.order_id = t2.order_id
GROUP BY 1
ORDER BY month;'''
result = sql_to_df(sql_13)
result

,month,total_orders,monthly_revenue,aov
0,2016-09-01,1,134.97,134.9700000000000000
1,2016-10-01,265,40325.11,152.1702264150943396
2,2016-12-01,1,10.90,10.9000000000000000
3,2017-01-01,750,111798.36,149.0644800000000000
4,2017-02-01,1653,234223.40,141.6959467634603751
5,2017-03-01,2546,359198.85,141.0836017282010998
6,2017-04-01,2303,340669.68,147.9243074250976987
7,2017-05-01,3546,489338.25,137.9972504230118443
8,2017-06-01,3135,421923.37,134.5848070175438596
9,2017-07-01,3872,481604.52,124.3813326446280992


**Key Insights**

- Olist experienced strong sales growth throughout 2017, with monthly revenue reaching its highest point in November 2017 (987,765), alongside the highest order volume (7,289 orders).
- In 2018, monthly revenue remained relatively high, reaching 977,545 in May, despite a lower order volume than November 2017.
- Average Order Value (AOV) remained relatively stable, fluctuating between approximately 124 and 152, suggesting that revenue growth was primarily driven by increased order volume rather than a consistent increase in average order value.


#### 15. Are Order Cancellations Increasing or Decreasing Over Time?


In [21]:
sql_14 = '''SELECT
    *,
    100.0 * cancelled / monthly_orders AS cancellation_rate,
    100.0 * unavailable / monthly_orders AS unavailable_rate
FROM (
    SELECT
        DATE_TRUNC('month', order_purchase_timestamp) AS month,
        COUNT(*) AS monthly_orders,
        COUNT(
            CASE
                WHEN order_status = 'canceled'
                THEN order_id
            END
        ) AS cancelled,
        COUNT(
            CASE
                WHEN order_status = 'unavailable'
                THEN order_id
            END
        ) AS unavailable
    FROM orders
    GROUP BY 1
) t
ORDER BY month;'''
result = sql_to_df(sql_14)
result

,month,monthly_orders,cancelled,unavailable,cancellation_rate,unavailable_rate
0,2016-09-01,4,2,0,50.0000000000000000,0E-20
1,2016-10-01,324,24,7,7.4074074074074074,2.1604938271604938
2,2016-12-01,1,0,0,0E-20,0E-20
3,2017-01-01,800,3,10,0.37500000000000000000,1.2500000000000000
4,2017-02-01,1780,17,45,0.95505617977528089888,2.5280898876404494
5,2017-03-01,2682,33,32,1.2304250559284116,1.1931394481730052
6,2017-04-01,2404,18,9,0.74875207986688851913,0.37437603993344425957
7,2017-05-01,3700,29,31,0.78378378378378378378,0.83783783783783783784
8,2017-06-01,3245,16,24,0.49306625577812018490,0.73959938366718027735
9,2017-07-01,4026,28,52,0.69547938400397416791,1.2916045702930949


**Key Insights**

- Order cancellation rates remained relatively low throughout most of the period, generally below 1.3%, with no consistent upward or downward trend.
- Cancellation rates increased to 1.09% in February 2018 and 1.29% in August 2018, indicating some fluctuations in order cancellation performance.
- The unavailable rate generally declined during 2018, reaching 0.06% in June and 0.11% in August, suggesting an improvement in product availability.
- September and October 2018 show unusually high cancellation rates (93.75% and 100%), but these figures are based on only 16 and 4 orders, respectively, and should be excluded from trend comparisons.


#### 16. Which States Have the Highest Order Cancellation Rates?


In [22]:
sql_15 = '''SELECT
    *,
    ROUND(
        100.0 * canceled_orders / total_orders,
        2
    ) AS cancellation_rate
FROM (
    SELECT
        customer_state,
        COUNT(order_id) AS total_orders,
        COUNT(
            CASE
                WHEN order_status = 'canceled'
                THEN order_id
            END
        ) AS canceled_orders
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    GROUP BY 1
) t
ORDER BY cancellation_rate DESC;'''
result = sql_to_df(sql_15)
result

,customer_state,total_orders,canceled_orders,cancellation_rate
0,RR,46,1,2.17
1,RO,253,3,1.19
2,PI,495,4,0.81
3,SP,41746,327,0.78
4,RJ,12852,86,0.67
5,GO,2020,13,0.64
6,MG,11635,64,0.55
7,MA,747,4,0.54
8,CE,1336,7,0.52
9,SC,3637,19,0.52


**Key Insights**

- Roraima (RR) has the highest cancellation rate at 2.17%, followed by Rondônia (RO) at 1.19%. However, both states have relatively low order volumes, so these rates should be interpreted cautiously.
- São Paulo (SP) accounts for the highest number of canceled orders (327), reflecting its significantly larger order volume of 41,746 orders.
- Overall, cancellation rates remain relatively low across most states, with no state exceeding 2.2%.


#### 17. How Efficiently Does Olist Deliver Orders to Customers?


In [23]:
sql_16 = '''WITH delivery_days AS (
    SELECT
        order_id,
        order_estimated_delivery_date,
        order_delivered_customer_date,
        DATE(order_estimated_delivery_date) -
        DATE(order_delivered_customer_date) AS delivery_days
    FROM orders
    WHERE order_estimated_delivery_date IS NOT NULL
        AND order_delivered_customer_date IS NOT NULL
        AND order_status = 'delivered'
),
deliver_status AS (
    SELECT
        *,
        CASE
            WHEN delivery_days = 0 THEN 'On Time'
            WHEN delivery_days > 0 THEN 'Early Delivery'
            WHEN delivery_days < 0 THEN 'Late Delivery'
        END AS delivery_status
    FROM delivery_days
)
SELECT
    delivery_status,
    COUNT(*) AS total_orders,
    100.0 * COUNT(*) / SUM(COUNT(*)) OVER() AS percentage
FROM deliver_status
GROUP BY 1
ORDER BY total_orders DESC;'''
result = sql_to_df(sql_16)
result

,delivery_status,total_orders,percentage
0,Early Delivery,88644,91.8876334611796413
1,Late Delivery,6534,6.7730900798175599
2,On Time,1292,1.3392764590027988


**Key Insights**

- 91.89% of orders were delivered earlier than the estimated delivery date, indicating that Olist generally delivers orders ahead of schedule.
- Only 6.77% of orders were delivered late, while 1.34% arrived exactly on the estimated date.
- Overall, 93.23% of orders were delivered on or before the estimated date, suggesting strong delivery performance.


#### 18. What Is the Delivery Performance by State?


In [24]:
sql_17 = '''WITH delivery_time AS (
    SELECT
        c.customer_state,
        DATE(o.order_estimated_delivery_date) -
        DATE(o.order_delivered_customer_date) AS delivery_days
    FROM orders o
    JOIN customers c
        ON o.customer_id = c.customer_id
    WHERE o.order_estimated_delivery_date IS NOT NULL
        AND o.order_delivered_customer_date IS NOT NULL
        AND o.order_status = 'delivered'
),
delivery_status AS (
    SELECT
        *,
        CASE
            WHEN delivery_days = 0 THEN 'On Time'
            WHEN delivery_days > 0 THEN 'Early Delivery'
            WHEN delivery_days < 0 THEN 'Late Delivery'
        END AS delivery_status
    FROM delivery_time
)
SELECT
    customer_state,
    delivery_status,
    COUNT(*) AS total_orders,
    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER(PARTITION BY customer_state),
        2
    ) AS percentage
FROM delivery_status
GROUP BY 1, 2
ORDER BY customer_state, delivery_status;'''
result = sql_to_df(sql_17)
result

,customer_state,delivery_status,total_orders,percentage
0,AC,Early Delivery,77,96.25
1,AC,Late Delivery,3,3.75
2,AL,Early Delivery,302,76.07
3,AL,Late Delivery,85,21.41
4,AL,On Time,10,2.52
...,...,...,...,...
72,SP,Late Delivery,1820,4.49
73,SP,On Time,567,1.40
74,TO,Early Delivery,239,87.23
75,TO,Late Delivery,27,9.85


**Key Insights**

- Delivery performance varies across states. Alagoas (AL) has the highest late delivery rate at 21.41%, followed by Maranhão (MA) at 17.43% and Sergipe (SE) at 15.22%.
- São Paulo (SP) and Minas Gerais (MG) show relatively strong delivery performance, with late delivery rates of 4.49% and 4.57%, respectively, despite having the highest order volumes.
- These differences suggest that delivery efficiency varies by region, highlighting opportunities to improve logistics performance in states with higher late delivery rates.


#### 19. How Many Customers Made Only One Purchase vs. Repeat Purchases?


In [25]:
sql_18 = '''SELECT
    COUNT(*) AS total_customers,
    COUNT(*) FILTER (WHERE total_orders = 1) AS single_order_customers,
    COUNT(*) FILTER (WHERE total_orders > 1) AS repeat_customers
FROM (
    SELECT
        c.customer_unique_id,
        COUNT(o.order_id) AS total_orders
    FROM orders o
    JOIN customers c
        ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
) t;'''
result = sql_to_df(sql_18)
result

,total_customers,single_order_customers,repeat_customers
0,93358,90557,2801


**Key Insights**

- 97.00% of customers (90,557 out of 93,358) made only one purchase, while just 3.00% made repeat purchases.
- The very low repeat purchase rate indicates a potential customer retention challenge and an opportunity to improve customer loyalty and encourage repeat purchases.


## 4. Final Results

- **Marketing acquisition:** Organic Search generated the largest share of leads (28.70%), while Paid Search had the highest conversion rate among the identifiable major channels (12.30%). Unknown sources represented 14.49% of leads.
- **Seller acquisition and activation:** Home Decor represented the largest share of acquired sellers (12.47%). However, only 45.13% of acquired sellers became active, and 44.66% fulfilled delivered customer orders.
- **Seller performance:** Watches generated the highest product revenue (115,901) despite having only 2 active sellers out of 8 acquired sellers. Health & Beauty recorded the highest delivered-order count (693).
- **Marketplace operations:** 97.02% of orders were delivered. Of delivered orders, 91.89% arrived before the estimated date and 6.77% arrived late. Late-delivery rates varied by state.
- **Customer purchasing behavior:** 97.00% of customers made only one purchase, while 3.00% made repeat purchases.

## 5. Recommendations

- Improve marketing-source tracking to reduce the 14.49% of leads recorded with an unknown source.
- Review Social Media lead targeting and conversion performance, given its 5.56% conversion rate despite generating a significant share of leads.
- Examine the seller onboarding and activation process, since 45.13% of acquired sellers became active on the marketplace.
- Review delivery performance in states with higher late-delivery rates, particularly Alagoas, Maranhão, and Sergipe.
- Investigate ways to encourage repeat purchases, considering that only 3.00% of customers made more than one purchase.